# demo

> Train the paper's flexible-policy agent and reproduce the shape of Fig 2d-e (Dan et al., Neuron 2024).

This notebook follows `specs/demo/`. **How to run:** [`specs/demo/RUN.md`](../specs/demo/RUN.md). Normal tests (`nbdev-test`) run a 5 s quick version so every plot still renders. The full version (`nbdev-test --flags slow`, or running the notebook by hand) trains for 240 s, averages 20 runs, and checks acceptance criteria AC-5 to AC-8.

Figures and metrics are written under `results/demo/` (gitignored; layout in RUN.md).

## Setup

Parameters come from Table 1 of the paper; the goal heading is 180 degrees.

In [ ]:
from math import pi
import numpy as np
import matplotlib.pyplot as plt
from explRL.params import Params
from explRL.basis import vonMisesBasis
from explRL.core import loadAgent, loadEnv
from explRL.train import (
    train_agent, evaluatePolicy, runMany,
    save_demo_figure, save_demo_metrics,
)

params = Params()
seed, goal, nRuns = 0, pi, 20
artifact_mode = 'quick'
params


In [ ]:
fk = vonMisesBasis(params.nb, params.nx, params.kappa)
deg = np.arange(params.nx) * 360 / params.nx
plt.figure(figsize=(8, 2))
plt.plot(deg, fk.T, lw=0.8)
plt.xlabel('heading (deg)'); plt.ylabel('basis value'); plt.title(f'{params.nb} von Mises basis functions')
plt.tight_layout()
save_demo_figure('basis', fig=plt.gcf(), mode=None)
plt.show()


## One training run

The quick run keeps normal tests fast; the slow cell replaces it with a full 240 s run.

In [ ]:
agent = loadAgent(params=params, goal=goal, seed=seed)
env = loadEnv(goal=goal)
hist = train_agent(agent, env, Ttot=5.0)


In [ ]:
#| slow
artifact_mode = 'full'
agent = loadAgent(params=params, goal=goal, seed=seed)
env = loadEnv(goal=goal)
hist = train_agent(agent, env, Ttot=params.Ttot)


Heading over time at the start and end of the run (red dots mark saccades; dashed line is the goal):

In [ ]:
t_sec = hist.t * params.delT
seg = min(20.0, t_sec[-1] / 2)
fig, axes = plt.subplots(1, 2, figsize=(10, 2.5), sharey=True)
for ax, (lo, hi), name in zip(axes, [(0, seg), (t_sec[-1] - seg, t_sec[-1])], ['first', 'last']):
    m = (t_sec >= lo) & (t_sec <= hi)
    ax.plot(t_sec[m], np.rad2deg(hist.theta[m]), 'k-', lw=0.6)
    ax.plot(t_sec[m & hist.saccade], np.rad2deg(hist.theta[m & hist.saccade]), 'r.', ms=3)
    ax.axhline(np.rad2deg(goal), ls='--', c='gray')
    ax.set_title(f'{name} {seg:.0f} s'); ax.set_xlabel('time (s)')
axes[0].set_ylabel('heading (deg)')
plt.tight_layout()
save_demo_figure('heading_trajectory', fig=fig, mode=artifact_mode)
plt.show()


The policy learned in this run, as a function of heading relative to the goal:

In [ ]:
ev = evaluatePolicy(agent)
rel = (deg - np.rad2deg(goal) + 180) % 360 - 180
order = np.argsort(rel)
fig, axes = plt.subplots(1, 3, figsize=(12, 2.5))
for ax, key, label in zip(axes, ['nu', 'fixDur', 'turnBias'], ['drift rate', 'mean fixation duration (s)', 'turn bias (+ = CW)']):
    ax.plot(rel[order], ev[key][order]); ax.set_xlabel('heading - goal (deg)'); ax.set_title(label)
plt.tight_layout()
save_demo_figure('learned_policy', fig=fig, mode=artifact_mode)
t_sec = hist.t * params.delT
save_demo_metrics(
    {
        'mode': artifact_mode,
        'seed': seed,
        'goal_rad': goal,
        'nRuns': 1,
        'Ttot_s': float(t_sec[-1] + params.delT * hist.dt[-1]),
    },
    mode=artifact_mode,
)
plt.show()


## Many runs (Fig 2d-e analogue)

Mean and standard error over independent runs with different seeds (the paper used 100 runs; set `nRuns` to reproduce that).

In [ ]:
#| slow
runs = runMany(params, goal, nRuns, seed=seed)
fix = np.stack([r['eval']['fixDur'] for r in runs])
bias = np.stack([r['eval']['turnBias'] for r in runs])
res = np.stack([r['residency'] for r in runs])
fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
for ax, data, label in zip(axes, [fix, bias], ['mean fixation duration (s)', 'turn bias (+ = CW)']):
    mean, sem = data.mean(0)[order], data.std(0)[order] / np.sqrt(len(runs))
    ax.plot(rel[order], mean); ax.fill_between(rel[order], mean - sem, mean + sem, alpha=0.3)
    ax.set_xlabel('heading - goal (deg)'); ax.set_title(label)
plt.tight_layout()
save_demo_figure('fixdur_turnbias', fig=fig, mode=artifact_mode)
plt.show()


Residency (fraction of time within ±45 degrees of the goal) across the run, averaged over runs:

In [ ]:
#| slow
plt.figure(figsize=(5, 2.5))
plt.plot(np.arange(1, res.shape[1] + 1), res.mean(0), 'o-')
plt.xlabel('time bin (of 8)'); plt.ylabel('residency near goal')
plt.tight_layout()
save_demo_figure('residency', fig=plt.gcf(), mode=artifact_mode)
plt.show()


## Acceptance checks (AC-5 to AC-8)

In [ ]:
#| slow
def cols(lo, hi):
    "Basis columns whose heading relative to the goal lies in [lo, hi] degrees."
    return (rel >= lo) & (rel <= hi)

fixM, biasM = fix.mean(0), bias.mean(0)
near, opposite = fixM[cols(-30, 30)].mean(), fixM[np.abs(rel) >= 150].mean()
ccw, cw = biasM[cols(-90, -10)].mean(), biasM[cols(10, 90)].mean()
early, late = res[:, :2].mean(), res[:, -2:].mean()
print(f'AC-5 fixation duration near goal {near:.3f} s vs opposite {opposite:.3f} s')
print(f'AC-6 turn bias CCW of goal {ccw:+.3f}, CW of goal {cw:+.3f}')
print(f'AC-7 residency first quarter {early:.3f}, last quarter {late:.3f}')
assert near > opposite                                    # AC-5
assert ccw > 0 > cw                                       # AC-6
assert late > early                                       # AC-7
assert all(np.isfinite(r['omega']).all() for r in runs)   # AC-8
save_demo_metrics(
    {
        'mode': artifact_mode,
        'seed': seed,
        'goal_rad': goal,
        'nRuns': nRuns,
        'Ttot_s': params.Ttot,
        'AC5_fixation_near_goal_s': near,
        'AC5_fixation_opposite_s': opposite,
        'AC6_turn_bias_ccw': ccw,
        'AC6_turn_bias_cw': cw,
        'AC7_residency_first_quarter': early,
        'AC7_residency_last_quarter': late,
        'AC8_all_weights_finite': True,
    },
    mode=artifact_mode,
)


## Summary

With the paper's Table 1 parameters and no tuning, 20 runs of 240 s (seeds 0-19) gave:

- **AC-5**: mean fixation duration 0.68 s within ±30 degrees of the goal, versus 0.11 s near the opposite heading.
- **AC-6**: turn bias +0.91 for headings 10-90 degrees counter-clockwise of the goal and -0.92 for headings 10-90 degrees clockwise of it, so saccades point toward the goal.
- **AC-7**: residency near the goal rose from 0.49 in the first quarter of the run to 0.78 in the last quarter.
- **AC-8**: all weights stayed finite.

This matches the qualitative shape of Fig 2d-e: long fixations at the goal and saccades biased toward it. The 20 runs take about 35 s on a laptop.